In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ---------------------------------------------
# 1. Read raw CSV from volume
# ---------------------------------------------
raw_path = "/Volumes/workspace/team_assignment/crime_data/Crime_Data_from_2020_to_Present_20251117.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path)
)

print("RAW ROW COUNT:", df_raw.count())

# ---------------------------------------------
# 2. Standardize column names (remove spaces, ()
# ---------------------------------------------
df_bronze = df_raw.toDF(*[c.strip().upper().replace(" ", "_").replace("(", "").replace(")", "") for c in df_raw.columns])

# ---------------------------------------------
# 3. Write Bronze Delta table
# ---------------------------------------------
df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.team_assignment.bronze_la_crime")

print("✔️ Bronze table created: workspace.team_assignment.bronze_la_crime")


In [0]:
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. Load Bronze Table
# ------------------------------------------------------------
df_bronze = spark.table("workspace.team_assignment.bronze_la_crime")

# ------------------------------------------------------------
# 2. Clean Column Names (Remove spaces, invalid chars)
# ------------------------------------------------------------
df_clean = df_bronze.toDF(*[
    c.strip().upper().replace(" ", "_").replace("(", "").replace(")", "")
    for c in df_bronze.columns
])

# ------------------------------------------------------------
# 3. Fix Timestamps (DATE_RPTD, DATE_OCC)
# These columns contain formats like: "2020 Nov 07 12:00:00 AM"
# ------------------------------------------------------------

timestamp_format = "yyyy MMM dd hh:mm:ss a"

df_silver = (
    df_clean
    .withColumn(
        "DATE_RPTD_TS",
        F.coalesce(
            F.to_timestamp("DATE_RPTD", timestamp_format),
            F.to_timestamp("DATE_RPTD"),            # fallback
            F.lit(None)
        )
    )
    .withColumn(
        "DATE_OCC_TS",
        F.coalesce(
            F.to_timestamp("DATE_OCC", timestamp_format),
            F.to_timestamp("DATE_OCC"),
            F.lit(None)
        )
    )
)

# Drop original raw string timestamps after converting
df_silver = df_silver.drop("DATE_RPTD").drop("DATE_OCC")

# ------------------------------------------------------------
# 4. Write Silver Table (No schema merge needed)
# ------------------------------------------------------------
df_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.team_assignment.silver_la_crime"
)

print("✅ Silver table created successfully: workspace.team_assignment.silver_la_crime")


In [0]:
df_silver = spark.table("workspace.team_assignment.silver_la_crime")
df_silver.printSchema()
df_silver.show(5)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_clean = (
    df_silver
    # --- convert DATE_RPTD ---
    .withColumn("DATE_RPTD_CLEAN", to_timestamp(col("DATE_RPTD"), "yyyy-MM-dd HH:mm:ss"))
    # fallback if format is like "2021 Apr 11 12:00:00 AM"
    .withColumn("DATE_RPTD_CLEAN",
                when(col("DATE_RPTD_CLEAN").isNull(),
                     to_timestamp(col("DATE_RPTD"), "yyyy MMM dd hh:mm:ss a"))
                .otherwise(col("DATE_RPTD_CLEAN"))
    )

    # --- convert DATE_OCC ---
    .withColumn("DATE_OCC_CLEAN", to_timestamp(col("DATE_OCC"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("DATE_OCC_CLEAN",
                when(col("DATE_OCC_CLEAN").isNull(),
                     to_timestamp(col("DATE_OCC"), "yyyy MMM dd hh:mm:ss a"))
                .otherwise(col("DATE_OCC_CLEAN"))
    )

    # --- convert TIME_OCC (int → HH:MM:SS) ---
    .withColumn("TIME_OCC_CLEAN",
                lpad(col("TIME_OCC").cast("string"), 4, "0"))
    .withColumn("TIME_OCC_CLEAN",
                concat_ws(":",
                    substring(col("TIME_OCC_CLEAN"), 1, 2),
                    substring(col("TIME_OCC_CLEAN"), 3, 2),
                    lit("00")
                )
    )

    # remove bad coordinates
    .withColumn("LON", col("LON").cast("double"))
    .withColumn("LAT", col("LAT").cast("double"))
)


In [0]:
df_final_silver = (
    df_clean
    .drop("DATE_RPTD", "DATE_OCC", "TIME_OCC")
    .withColumnRenamed("DATE_RPTD_CLEAN", "DATE_RPTD")
    .withColumnRenamed("DATE_OCC_CLEAN", "DATE_OCC")
    .withColumnRenamed("TIME_OCC_CLEAN", "TIME_OCC")
)


In [0]:
# Load silver table
df_silver = spark.table("workspace.team_assignment.silver_la_crime")

# -------------------------------
# 1. Final cleanup transformations
# -------------------------------

df_final_silver = (
    df_silver
        .withColumn("DATE_RPTD_CLEAN", col("DATE_RPTD_TS"))
        .withColumn("DATE_OCC_CLEAN", col("DATE_OCC_TS"))
        .drop("DATE_RPTD", "DATE_OCC")           # if exist
        .drop("DATE_RPTD_TS", "DATE_OCC_TS")     # old columns not needed
)

# -------------------------------
# 2. Write final clean silver table
# -------------------------------

df_final_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.silver_la_crime_clean")

print("✅ Final Clean Silver Table Created: workspace.team_assignment.silver_la_crime_clean")


In [0]:
df = spark.table("workspace.team_assignment.silver_la_crime")
df.printSchema()


In [0]:
from pyspark.sql.functions import *

df_date = (
    df.select("DATE_RPTD_TS")
      .withColumnRenamed("DATE_RPTD_TS", "full_date")
      .withColumn("date_key", date_format("full_date", "yyyyMMdd"))
      .withColumn("year", year("full_date"))
      .withColumn("month", month("full_date"))
      .withColumn("quarter", quarter("full_date"))
      .withColumn("day_of_month", dayofmonth("full_date"))
      .withColumn("day_of_week", dayofweek("full_date"))
      .withColumn("week_of_year", weekofyear("full_date"))
      .withColumn("is_weekend", (dayofweek("full_date").isin(1,7)).cast("int"))
      .distinct()
)

df_date.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_date")


In [0]:
df_time = (
    df.select("TIME_OCC")
      .withColumn("time_key", col("TIME_OCC"))
      .withColumn("time_of_day", col("TIME_OCC"))
      .withColumn("hour_24h", floor(col("TIME_OCC")/100))
      .withColumn("minute", col("TIME_OCC") % 100)
      .withColumn("time_bucket",
                  when(col("TIME_OCC") < 600, "Night")
                  .when(col("TIME_OCC") < 1200, "Morning")
                  .when(col("TIME_OCC") < 1800, "Afternoon")
                  .otherwise("Evening"))
      .distinct()
)

df_time.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_time")


In [0]:
df_area = (
    df.select("AREA", "AREA_NAME")
      .withColumnRenamed("AREA", "area_key")
      .withColumnRenamed("AREA_NAME", "area_name")
      .distinct()
)

df_area.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_area")


In [0]:
df_crime = (
    df.select("CRM_CD", "CRM_CD_1", "CRM_CD_2", "CRM_CD_3", "CRM_CD_4", "Crm_Cd_Desc")
      .withColumnRenamed("CRM_CD", "crime_key")
      .withColumnRenamed("Crm_Cd_Desc", "crime_desc")
      .distinct()
)

df_crime.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_crime")


In [0]:
df_location = (
    df.select("LOCATION", "CROSS_STREET", "LAT", "LON")
      .withColumn("location_key", monotonically_increasing_id())
      .distinct()
)

df_location.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_location")


In [0]:
df_victim = (
    df.select("VICT_AGE", "VICT_SEX", "VICT_DESCENT")
      .withColumn("victim_key", monotonically_increasing_id())
      .distinct()
)

df_victim.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_victim")


In [0]:
df_status = (
    df.select("STATUS", "STATUS_DESC")
      .withColumnRenamed("STATUS", "status_key")
      .distinct()
)

df_status.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_status")


In [0]:
df_weapon = (
    df.select("WEAPON_USED_CD", "WEAPON_DESC")
      .withColumnRenamed("WEAPON_USED_CD", "weapon_key")
      .distinct()
)

df_weapon.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_weapon")


In [0]:
df_rpt_dist = (
    df.select("Rpt_Dist_No", "AREA")
      .withColumnRenamed("Rpt_Dist_No", "rpt_district_key")
      .withColumnRenamed("AREA", "area_id")
      .distinct()
)

df_rpt_dist.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_reporting_district")


In [0]:
df_mo = (
    df.select("MOCODES")
      .withColumnRenamed("MOCODES", "mo_key")
      .distinct()
)

df_mo.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.team_assignment.dim_mo")


In [0]:
# Load cleaned silver table
df = spark.table("workspace.team_assignment.silver_la_crime_clean")

# Load all dimension tables
dim_date      = spark.table("workspace.team_assignment.dim_date")
dim_time      = spark.table("workspace.team_assignment.dim_time")
dim_area      = spark.table("workspace.team_assignment.dim_area")
dim_crime     = spark.table("workspace.team_assignment.dim_crime")
dim_weapon    = spark.table("workspace.team_assignment.dim_weapon")
dim_victim    = spark.table("workspace.team_assignment.dim_victim")
dim_status    = spark.table("workspace.team_assignment.dim_status")
dim_location  = spark.table("workspace.team_assignment.dim_location")
dim_rd        = spark.table("workspace.team_assignment.dim_reporting_dist")


In [0]:
fact = (
    df.alias("f")

    # DATE REPORTED
    .join(dim_date.alias("dr"),
          F.col("f.DATE_RPTD_TS") == F.col("dr.full_date"),
          "left")

    # DATE OCCURRED
    .join(dim_date.alias("do"),
          F.col("f.DATE_OCC_TS") == F.col("do.full_date"),
          "left")

    # TIME OCCURRED
    .join(dim_time.alias("ti"),
          F.col("f.TIME_OCC") == F.col("ti.time_occd"),
          "left")

    # AREA
    .join(dim_area.alias("ar"),
          F.col("f.AREA") == F.col("ar.area_id"),
          "left")

    # CRIME TYPE
    .join(dim_crime.alias("cr"),
          F.col("f.CRM_CD") == F.col("cr.crime_cd1"),
          "left")

    # WEAPON
    .join(dim_weapon.alias("we"),
          F.col("f.WEAPON_USED_CD") == F.col("we.weapon_used_cd"),
          "left")

    # VICTIM INFO
    .join(dim_victim.alias("vi"),
          (F.col("f.VICT_AGE") == F.col("vi.vict_age")) &
          (F.col("f.VICT_SEX") == F.col("vi.vict_sex")) &
          (F.col("f.VICT_DESCENT") == F.col("vi.vict_descent")),
          "left")

    # STATUS
    .join(dim_status.alias("st"),
          F.col("f.STATUS") == F.col("st.status_code"),
          "left")

    # LOCATION
    .join(dim_location.alias("lo"),
          (F.col("f.LAT") == F.col("lo.lat")) &
          (F.col("f.LON") == F.col("lo.lon")),
          "left")

    # REPORTING DISTRICT
    .join(dim_rd.alias("rd"),
          F.col("f.RPT_DIST_NO") == F.col("rd.rpt_district_id"),
          "left")
)


In [0]:
fact_final = fact.select(
    F.col("f.DR_NO").alias("crime_incident_key"),

    F.col("dr.date_key").alias("date_rpt_key"),
    F.col("do.date_key").alias("date_occ_key"),
    F.col("ti.time_key").alias("time_occ_key"),

    F.col("ar.area_key"),
    F.col("cr.crime_key"),
    F.col("we.weapon_key"),
    F.col("vi.victim_key"),
    F.col("st.status_key"),
    F.col("lo.location_key"),
    F.col("rd.rpt_district_key"),

    # Any useful measures
    F.col("f.CRM_CD").alias("crime_code"),
    F.col("f.WEAPON_USED_CD").alias("weapon_used_code"),
    F.col("f.PREMIS_CD").alias("premise_code")
)


In [0]:
df = spark.table("workspace.team_assignment.silver_la_crime_clean")
df.printSchema()


In [0]:
df = spark.table("workspace.team_assignment.silver_la_crime_clean")

# Build Fact Table
fact = (
    df.select(
        df.DR_NO.alias("crime_incident_key"),

        # Date keys
        date_format("DATE_RPTD_CLEAN", "yyyyMMdd").cast("int").alias("date_rpt_key"),
        date_format("DATE_OCC_CLEAN", "yyyyMMdd").cast("int").alias("date_occ_key"),

        # Time key
        df.TIME_OCC.alias("time_occ_key"),

        # Area & district
        df.AREA.alias("area_key"),
        df.RPT_DIST_NO.alias("rpt_district_key"),

        # Crime key
        df.CRM_CD_1.alias("crime_key"),

        # Weapon key
        df.WEAPON_USED_CD.alias("weapon_key"),

        # Status key
        df.STATUS.alias("status_key"),

        # Location key (natural key)
        df.LOCATION.alias("location_key"),

        # Victim key (composite — leave natural values)
        df.VICT_AGE.alias("vict_age"),
        df.VICT_SEX.alias("vict_sex"),
        df.VICT_DESCENT.alias("vict_descent"),
    )
)

# Save delta table
fact.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.team_assignment.fact_crime_incident"
)

print("✅ Fact table created: workspace.team_assignment.fact_crime_incident")


In [0]:
spark.table("workspace.team_assignment.fact_crime_incident").show(5)

spark.table("workspace.team_assignment.dim_date").show(5)
spark.table("workspace.team_assignment.dim_time").show(5)
spark.table("workspace.team_assignment.dim_area").show(5)
spark.table("workspace.team_assignment.dim_crime").show(5)
spark.table("workspace.team_assignment.dim_victim").show(5)
spark.table("workspace.team_assignment.dim_status").show(5)
spark.table("workspace.team_assignment.dim_location").show(5)
spark.table("workspace.team_assignment.dim_reporting_district").show(5)
spark.table("workspace.team_assignment.dim_weapon").show(5)
spark.table("workspace.team_assignment.dim_mo").show(5)


In [0]:
fact = spark.table("workspace.team_assignment.fact_crime_incident")
dim_date = spark.table("workspace.team_assignment.dim_date")

fact.join(dim_date, fact.date_rpt_key == dim_date.date_key, "left_anti").count()
